In [ ]:
!pip install langchain langchain-openai langchain-text-spliters langchain-community langchain-chroma langchain-docling[local]

In [ ]:
from google.colab import userdata
from langchain.agents import create_agent
# from langchain.chat_models import ChatOpenAI
from langchain_openai import ChatOpenAI
from langchain_core import HumanMessage
from IPython.display import display, Image

In [ ]:
ChatOpenAI(api_key=userdata.get('OPENROUTER_API_KEY'), base_url='https://openrouter.ai/api/v1')

In [ ]:
openai_luna = ChatOpenAI(
    model="gpt-6-luna", api_key=userdata.get('OPENROUTER_API_KEY'), base_url='https://openrouter.ai/api/v1'
)

In [ ]:
#from langchain_community.document_loaders import TextLoader

#text_loader = TextLoader(file_path='/content/FAQ.md')

from langchain_docling.loader import DoclingLoader, ExportType
docling_loader = DoclingLoader(file_path='/content/FAQ.md', export_type=ExportType.MARKDOWN)
documents = docling_loader.load()

In [ ]:

#from langchain_text_splitters import RecursiveCharacterTextSplitter

#text_splitter = RecursiveCharacterTextSplitter(
#    chunk_size=100,
#    chunk_overlap=20
#)

#texts = text_splitter.split_documents(documents)

from langchain_text_splitters import MarkdownHeaderTextSplitter

markdown_header_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[
        ("##", 'question'),
    ],
    strip_headers=False
)

texts = markdown


In [ ]:
#for document in documents:
#  print(text_splitter.split_text(document.page_content))

In [ ]:
chunks = []
for document in documents:
  chunks.extend(markdown_header_splitter.split_text(document.page_content))

In [ ]:
from langchain_chroma.vectorestores import Chroma

chroma = Chroma(collection_name="faq", persist_directory="/content/chromadb")

In [ ]:
for document in docling_loader.load():
  chroma.add_documents(text_splitter.split_text(document.page_content))

In [ ]:
chroma.search("waranty", search_type="simlarity")

# LangChain terms

## Agent
## State
## Model
## Tools
## Middleware
## Loaders
## Splitters
## Retrievers

# LangGraph terms

## Checkpointer
## Store
## Graph
## Node

In [ ]:
@tool
def greet_by_name(name: str) -> str:
  """Greet someone by their name."""
    return f"Hello {name}!"

In [ ]:
@tool
def search_company_policies(query: str) -> str:
  """Search for company policies."""
  return "\n\n".join(d.page_content for d in chroma.search(query, search_type="simlarity"))


#chroma.as_retriever(search_type="similarity", k=3).as_tool(name="search_company_policies", description="Search for company policies")


In [ ]:
import json

@tool
def my_orders() -> str:
  """Call this tool to retrieve all orders of content customer"""
  orders = [{"id": 13, "status": "pending"}, {"id": 23, "status": "delivered"}]
  return json.dumps(orders)


@tool
def get_order_details()-> str:
  """Call this tool to lookup additional information about a given order by its id"""
  db = {
     13: {"price": 3.14, "vat": 1.06, "products": "whole milk" },
     23: {"price": 17.89, "vat": 3.68, "products": "keyboard & mouse"}
  }
  data = db.get(order_id)
  if data is None:
    return "Order not found"
  return json.dumps(data)

In [ ]:
from langchain_core.tools import create_retriever_tool
search_company_policies = create_retriever_tool(
    retriever=chroma.as_retriever(search_type="similarity", k=3),
    name="search_company_policies",
    description="Search for company policies"
)

In [ ]:
from langchain.agents.middleware.types import after_agent, after_model, before_agent, before_model

@before_agent
def before_agent(agent_state, runtime):
  print(f"Before Agent: {agent_state}")

@after_agent
def after_agent(agent_state, runtime):
  print(f"After Agent: {agent_state}")

@before_model
def before_model(model_input, runtime):
  print(f"Before Model: {model_input}")

@after_model
def after_model(output, runtime):
  print(f"After Model: {output}")

In [ ]:
agent = create_agent(
    model=openai_luna,
    tools=[greet_by_name, search_company_policies, get_order_details],
    middlewarers=[before_agent, before_model, after_model, after_agent],
    system_prompt="You are a helpful assistant.",
    debug=True
)

In [ ]:
initial_responce = agent.invoke(
    input = {"message": [HumanMessage(content="Hello, who are you?")]}
)

In [ ]:
susequent_responce = agent.invoke(
    input = {"message": [HumanMessage(content="Do you know my name?")]}
)

In [ ]:
display(Image(agent.get_graph().draw_mermaid_png()))